# 12 — Adaptive policy, CSC merge và sampling report

Notebook này nối các artifact của notebook 09–11 thành một phiên kiểm tra adaptive có thể chạy lại. Nó có ba phần: chọn policy trên validation, tạo/kiểm tra adaptive author/LoRA merge trên evaluation, và in cùng một bảng cho greedy, fixed SC/CSC và adaptive SC/CSC. `adaptive-sweep` chỉ dùng validation; `adaptive-csc-eval` dùng policy đã chọn để replay fixed prefixes với `offline_prefix_exact`. Kết quả online cuối cùng vẫn lấy từ `sampling-final`/`suite`, không dùng nhãn evaluation để chọn threshold.

Chạy notebook 09 trước để có generator/execution/adaptive cache, notebook 10 để có adapter, notebook 11 để freeze và chạy sampling matrix. Nếu các report đã tồn tại, notebook này có thể chạy ở chế độ report-only mà không tải model.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

configured = os.environ.get('SLM_PRJ_PATH')
candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/SLM-PRJ')]
REPO_DIR = next((p.resolve() for p in candidates if (p / 'research/slm_sql').is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git'), str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
if os.environ.get('INSTALL_CSC_DEPS', '0') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REPO_DIR / 'requirements-csc-inference.txt')], check=True)


In [ ]:
from scripts.csc_notebook import configure, run

CONFIG, cfg = configure(REPO_DIR)
SEED = int(os.environ.get('CSC_SEED', '42'))
SEEDS = [int(value) for value in os.environ.get('CSC_EVAL_SEEDS', str(SEED)).split(',') if value.strip()]
RUN_GPU = os.environ.get('RUN_CSC_GPU', '0') == '1'
RUN_POLICY_SWEEP = os.environ.get('RUN_ADAPTIVE_POLICY_SWEEP', '0') == '1'
RUN_ADAPTIVE_MERGE = os.environ.get('RUN_ADAPTIVE_MERGE', '0') == '1'
experiment = Path(cfg['experiment_dir'])
reports_dir = experiment / 'reports'
print({
    'config': str(CONFIG),
    'experiment': str(experiment),
    'seed': SEED,
    'seeds': SEEDS,
    'checkpoints': cfg['adaptive']['checkpoints'],
    'evaluation_k_values': cfg['evaluation']['sampling_k_values'],
    'run_gpu': RUN_GPU,
    'run_policy_sweep': RUN_POLICY_SWEEP,
    'run_adaptive_merge': RUN_ADAPTIVE_MERGE,
})
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))


## 1. Chọn policy trên validation

`adaptive-sweep` cần execution cache và fixed author/LoRA merge artifacts cho các checkpoint adaptive. Vì merger là model GPU, cell dưới có thể tạo các artifact còn thiếu khi bật `RUN_ADAPTIVE_POLICY_SWEEP=1` và `RUN_CSC_GPU=1`; lệnh sweep sau đó vẫn chạy CPU. Policy được ghi vào `adaptive/selected_policy.json` cùng hash của validation execution và merge artifacts. Không chạy sweep trên evaluation.


In [ ]:
if RUN_POLICY_SWEEP:
    if RUN_GPU:
        for k_value in cfg['adaptive']['checkpoints']:
            run(REPO_DIR, CONFIG, 'merge', '--split', 'validation', '--seed', str(SEED), '--variant', 'author', '--k', str(k_value))
            run(REPO_DIR, CONFIG, 'merge', '--split', 'validation', '--seed', str(SEED), '--variant', 'lora', '--k', str(k_value))
    else:
        print('Bỏ qua việc tạo validation merge mới; adaptive-sweep vẫn chạy CPU trên artifacts đã có.')
    run(REPO_DIR, CONFIG, 'adaptive-sweep', '--split', 'validation', '--seed', str(SEED))
else:
    selected = experiment / 'adaptive' / 'selected_policy.json'
    print('Policy artifact:', selected if selected.is_file() else 'missing; set RUN_ADAPTIVE_POLICY_SWEEP=1 after validation merge artifacts are ready')


## 2. Adaptive merge và offline policy evaluation

`adaptive-merge --variant author|lora` chạy merger trên **online adaptive cache** đã tạo bởi notebook 09/11. `adaptive-csc-eval` là phép replay offline dùng policy đã chọn từ validation và fixed-prefix merge checkpoints; nó kiểm tra provenance và freeze trước khi chạy trên evaluation. Hai báo cáo có accuracy basis khác nhau nên được hiển thị tách biệt.


In [ ]:
if RUN_ADAPTIVE_MERGE:
    if RUN_GPU:
        for seed in SEEDS:
            run(REPO_DIR, CONFIG, 'adaptive-merge', '--split', 'evaluation', '--seed', str(seed), '--variant', 'author')
            run(REPO_DIR, CONFIG, 'adaptive-merge', '--split', 'evaluation', '--seed', str(seed), '--variant', 'lora')
    else:
        print('Bỏ qua việc tạo adaptive merge mới; nếu artifacts đã có, các report vẫn có thể được đọc ở dưới.')
    # Offline replay là scoring CPU, nên có thể chạy với RUN_CSC_GPU=0 khi
    # fixed merge checkpoints và final_protocol đã được tạo ở session trước.
    for seed in SEEDS:
        run(REPO_DIR, CONFIG, 'adaptive-csc-eval', '--split', 'evaluation', '--seed', str(seed))
else:
    print('Adaptive merge/eval commands are disabled. Set RUN_ADAPTIVE_MERGE=1 to execute them.')


## 3. Bảng sampling cuối

Bảng dưới lấy từ `sampling.comparison` đã tạo bởi notebook 11 hoặc `sampling-final`. Với mỗi fixed `k`, nó in greedy, SC@k, author CSC@k, LoRA CSC@k và hai adaptive CSC conditions. `adaptive-csc-eval` được in riêng vì đó là replay `offline_prefix_exact`; không gộp nó vào bảng online.


In [ ]:
def print_sampling_report(path, k_values):
    if not path.is_file():
        print('Pending report:', path)
        return
    payload = json.loads(path.read_text())
    methods = payload.get('summary', {}).get('methods', {})
    print(f'\nReport: {path.name} | mode={payload.get("mode")} | seed={payload.get("seed")}')
    for k_value in k_values:
        print(f'\nK={k_value}')
        for method, label in [
            ('greedy', 'greedy'),
            (f'sc@{k_value}', f'SC@{k_value}'),
            (f'author_csc@{k_value}', f'author CSC@{k_value}'),
            (f'lora_csc@{k_value}', f'LoRA CSC@{k_value}'),
            ('adaptive_sc', 'adaptive SC'),
            ('adaptive_author_csc', 'adaptive + author CSC'),
            ('adaptive_lora_csc', 'adaptive + LoRA CSC'),
        ]:
            metrics = methods.get(method)
            if metrics is None:
                continue
            print({
                'method': label,
                'ex': metrics.get('ex'),
                'mean_samples': metrics.get('mean_samples'),
                'mean_tokens': metrics.get('mean_tokens'),
                'csc_calls': metrics.get('csc_calls'),
            })

for seed in SEEDS:
    print_sampling_report(reports_dir / f'evaluation-seed{seed}.sampling.comparison.json', cfg['evaluation']['sampling_k_values'])
    offline = reports_dir / f'evaluation-seed{seed}.adaptive-csc-eval.json'
    if offline.is_file():
        payload = json.loads(offline.read_text())
        print(f'\nOffline adaptive policy evaluation: {offline.name}')
        for method, metrics in payload.get('summary', {}).items():
            print({
                'method': method,
                'ex': metrics.get('ex'),
                'mean_samples': metrics.get('mean_samples'),
                'mean_tokens': metrics.get('mean_tokens'),
                'csc_calls': metrics.get('csc_calls'),
                'accuracy_basis': payload.get('accuracy_basis'),
            })

aggregate = reports_dir / 'evaluation.sampling.comparison.multi-seed.json'
if aggregate.is_file():
    payload = json.loads(aggregate.read_text())
    print('\nMulti-seed report:', aggregate)
    for method, metrics in payload.get('summary', {}).items():
        print(method, {key: metrics.get(key) for key in ('ex_mean', 'ex_std', 'seed_averaged_question_ci95', 'mean_samples', 'mean_tokens')})
else:
    print('No multi-seed comparison report yet:', aggregate)
